# AQI Predictor - Exploratory Data Analysis

Four short sections over the Karachi historical feature store: trend + missing-data, seasonality, category distribution, and feature correlation against the served model's SHAP global importance. Each chart is produced by `aqi_predictor.dashboard.eda_charts` - the exact same functions the dashboard's **Data Insights** page uses, so this notebook and the app never disagree about what the data shows.

In [1]:
from aqi_predictor.config import LOCATIONS, POLLUTANT_VARS, WEATHER_VARS
from aqi_predictor.dashboard import eda_charts
from aqi_predictor.feature_pipeline import store
from aqi_predictor.training_pipeline import registry

LOCATION = LOCATIONS[0]["name"]
df = store.get_feature_view("2000-01-01", "2100-01-01", locations=[LOCATION])
print(f"{LOCATION}: {len(df):,} rows, {df['time'].min()} .. {df['time'].max()}")

D:\AQI_Predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-06 02:25:30,755 INFO: Initializing external client


2026-09-06 02:25:30,758 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


2026-09-06 02:25:35,875 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/43151


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (47.84s) 
karachi: 18,456 rows, 2024-07-29 00:00:00+00:00 .. 2026-09-05 23:00:00+00:00


## 1. Trend & missing data

The full `us_aqi` history, with any missing-hour gaps shaded red underneath the line (gaps the feature pipeline couldn't interpolate - runs longer than the 3h limit in `features.py`).

In [2]:
eda_charts.trend_and_gaps(df)

alt.Chart(...)

## 2. Seasonality

Mean `us_aqi` by hour of day (UTC) and by month of year, side by side.

In [3]:
eda_charts.seasonality_patterns(df)

alt.HConcatChart(...)

## 3. Category distribution

How many observed hours fall in each US AQI category, coloured per `aqi_scale.py`'s official breakpoints.

In [4]:
eda_charts.category_distribution(df)

alt.Chart(...)

## 4. Feature correlation vs. SHAP global importance

Correlation heatmap among the core pollutant/weather columns, then a quick check: do the base features most correlated with `us_aqi` line up with what the currently-served `us_aqi_next` model's SHAP global importance actually ranks highest?

In [5]:
eda_charts.feature_correlation(df)

alt.Chart(...)

In [6]:
_model, meta = registry.load_best_model("us_aqi_next")
shap_importance = meta.get("shap_importance") or {}

base_cols = [c for c in [*POLLUTANT_VARS, *WEATHER_VARS] if c in df.columns]
corr_with_target = (
    df[base_cols].corr()["us_aqi"].drop("us_aqi").sort_values(key=abs, ascending=False)
)
print("Top base-feature |correlation| with us_aqi:")
print(corr_with_target.head(6).round(3).to_string())

algo = meta.get("metrics", {}).get("algorithm", "?")
print(f"\nServed us_aqi_next model: {algo} v{meta['version']}")
if shap_importance:
    shap_top = sorted(shap_importance.items(), key=lambda kv: kv[1], reverse=True)[:6]
    print("Top SHAP global importance (mean |SHAP|):")
    for feat, val in shap_top:
        print(f"  {feat:<28}{val:.3f}")
else:
    print("Served model has no shap_importance recorded.")

Using cached model files at 'D:\tmp\hopsworks\models\Aqiii\us_aqi_next\4\us_aqi_next_4'. Pass local_path or call Model.clear_cache(...) to force a fresh download.


Top base-feature |correlation| with us_aqi:
pm2_5               0.726
sulphur_dioxide     0.516
carbon_monoxide     0.459
surface_pressure    0.439
wind_speed_10m     -0.401
nitrogen_dioxide    0.368

Served us_aqi_next model: random_forest v4
Top SHAP global importance (mean |SHAP|):
  us_aqi                      15.539
  pm2_5_roll_mean_24h         2.486
  us_aqi_lag_1h               0.273
  ozone                       0.148
  us_aqi_change_1h            0.096
  nitrogen_dioxide            0.044
